# Turbofan RUL — Results

Predicting remaining useful life for NASA C-MAPSS FD001 engines. This notebook runs the
packaged pipeline and walks through what each step shows. The exploratory work it was
ported from is in `course_original.ipynb`.

Run `python run.py --subset FD001` first if you want the saved CSVs and figures; this
notebook recomputes everything it displays.

In [1]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd

from turbofan_rul import data, evaluate, models, plots

DATA_DIR = "../data"
FIGURES_DIR = "../figures"

## 1. The data

100 engines run to failure, every cycle recorded across 21 sensors and three operational
settings. The target is built per engine as `max(Cycle) - Cycle` and then clipped at 150:
an engine 300 cycles from failure is sensor-indistinguishable from one at 200, so an
uncapped target asks the model to learn a distinction the data does not contain.

Eight columns are dropped as dead — seven detected with `nunique() == 1`, plus `S6`,
which has two values but spends 98% of rows on one of them. `ID` and `Cycle` are held out
of the features too: `ID` is an arbitrary label, and `Cycle` leaks the answer because RUL
is computed from it.

In [2]:
train, test, features = data.load_dataset(DATA_DIR, "FD001", clip=150)

print(f"train: {train.shape[0]:,} rows, {train['ID'].nunique()} engines")
print(f"test:  {test.shape[0]} engines (last observed cycle of each)")
print(f"features ({len(features)}): {', '.join(features)}")
train.head()

train: 20,631 rows, 100 engines
test:  100 engines (last observed cycle of each)
features (16): Op S1, Op S2, S2, S3, S4, S7, S8, S9, S11, S12, S13, S14, S15, S17, S20, S21


,ID,Cycle,Op S1,Op S2,S2,S3,S4,S7,S8,S9,S11,S12,S13,S14,S15,S17,S20,S21,RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,150
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,150
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,150
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,150
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,150


In [3]:
# Failure times across the training fleet: how long these engines actually lasted.
train.groupby("ID")["Cycle"].max().describe()

count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: Cycle, dtype: float64

## 2. The linear baseline, and what clipping did

Uncapped, the linear model looked like it was overfitting badly. The two panels below are
the same model on the same held-out engines, differing only in whether the target was
clipped.

The left panel shows the problem was never variance: the cloud *curves* away from the
diagonal and flattens above ~180 cycles, where the sensors stop separating engines. That
flat region is where the error was coming from. It also pushes predictions below zero —
negative remaining life, which is physically impossible and the clearest sign that a plane
is being fitted to a curve.

In [4]:
uncapped_train, _, _ = data.load_dataset(DATA_DIR, "FD001", clip=None)

uncapped = evaluate.out_of_fold_predictions(models.linear_regression(), uncapped_train, features)
capped = evaluate.out_of_fold_predictions(models.linear_regression(), train, features)

plots.plot_clip_comparison(uncapped, capped, figures_dir=FIGURES_DIR)

PosixPath('../figures/linear_regression_clip.png')

In [5]:
for label, frame in [("uncapped", uncapped_train), ("capped at 150", train)]:
    folds = evaluate.cross_validate_model(models.linear_regression(), frame, features)
    rmse = folds.groupby("split")["rmse"].mean()
    print(
        f"{label:>14}: train {rmse['train']:.2f}  validation {rmse['validation']:.2f}  "
        f"gap {rmse['validation'] - rmse['train']:.2f}"
    )

_, capped_predictions = capped
negative = (capped_predictions < 0).sum()
print(f"\nnegative RUL predictions: {negative} of {len(capped_predictions):,} held-out rows")

      uncapped: train 44.63  validation 44.88  gap 0.26
 capped at 150: train 27.28  validation 27.50  gap 0.22

negative RUL predictions: 125 of 4,120 held-out rows


Averaged over five folds the train/validation gap is essentially zero in both cases. A
single 80/20 split showed a 15-cycle gap only because that particular validation set held
more of the unlearnable high-RUL region. Either way the conclusion is the same: this model
is bias-limited, so there is nothing for regularization to fix — which is why none was
applied.

## 3. Cross-validation, grouped by engine

Consecutive cycles from one engine are near-duplicates, so splitting on rows would train
on cycle 100 and validate on cycle 101. Every split here is `GroupKFold` on engine ID, so
an engine is wholly in training or wholly held out.

In [6]:
cv_metrics = evaluate.cross_validate_all(models.all_models(), train, features, n_splits=5)
cv_metrics[cv_metrics["split"] == "validation"][
    ["model", "rmse_mean", "rmse_std", "mae_mean", "r2_mean"]
].round(3)

,model,rmse_mean,rmse_std,mae_mean,r2_mean
1,Linear Regression,27.497,1.893,22.424,0.696
3,Random Forest,24.600,0.764,18.743,0.757
5,Gradient Boosting,24.607,0.798,18.696,0.757


In [7]:
plots.plot_model_comparison(cv_metrics, figures_dir=FIGURES_DIR)

PosixPath('../figures/model_comparison.png')

Both tree models beat the linear baseline by about 3 cycles. They do not beat each other:
24.60 against 24.61, with fold standard deviations of about 0.8. The error bars overlap
almost completely, and the honest reading is a tie.

## 4. Which sensors carry the signal

A quick look at what the forest actually leans on.

In [8]:
forest = models.random_forest().fit(train[features], train["RUL"])
importance = pd.Series(forest.feature_importances_, index=features)
importance.sort_values(ascending=False).head(10).round(4)

S11    0.6433
S9     0.1451
S4     0.0663
S12    0.0399
S7     0.0199
S14    0.0189
S15    0.0133
S13    0.0089
S21    0.0089
S2     0.0078
dtype: float64

The top features are physically sensible: HPC outlet static pressure (S11), core speed
(S9) and LPT outlet temperature (S4) — FD001's injected fault is high-pressure compressor
degradation, so the model is keying on the compressor and the turbine behind it.

## 5. Complexity sweeps

One knob per tree model, five folds at every setting. Slow — boosting refits up to 1000
trees five times over.

In [9]:
rf_sweep = evaluate.sweep_parameter(
    models.random_forest, "max_depth", models.RF_DEPTH_GRID, train, features
)
gb_sweep = evaluate.sweep_parameter(
    models.gradient_boosting, "n_estimators", models.GB_ESTIMATOR_GRID, train, features
)

display(rf_sweep.drop(columns="x").round(3))
display(gb_sweep.drop(columns="x").round(3))

plots.plot_complexity_sweeps(rf_sweep, gb_sweep, figures_dir=FIGURES_DIR)

,max_depth,train_rmse_mean,train_rmse_std,val_rmse_mean,val_rmse_std
0,3,27.869,0.429,28.438,0.974
1,5,25.019,0.211,26.028,0.942
2,8,22.280,0.156,24.785,0.752
3,10,20.022,0.167,24.600,0.764
4,15,13.522,0.261,24.665,0.770
5,20,10.004,0.181,24.732,0.793
6,None,8.951,0.052,24.774,0.787


,n_estimators,train_rmse_mean,train_rmse_std,val_rmse_mean,val_rmse_std
0,50,23.528,0.161,24.760,0.784
1,100,23.119,0.167,24.627,0.770
2,200,22.608,0.183,24.607,0.798
3,500,21.357,0.187,24.758,0.823
4,1000,19.723,0.210,25.013,0.876


PosixPath('../figures/complexity_sweeps.png')

Two different shapes, and the difference is the mechanism.

The **forest** overfits enormously at full depth — training RMSE drops to about 9 cycles
while validation holds at 24.8 — and validation does not care. It is flat from depth 8
onward, because bagging averages 100 independently-overfitted trees and their errors
cancel. Depth 10 is chosen as the start of that plateau: the cheapest model that gives up
nothing.

**Boosting** turns back up. Each tree fits the previous ensemble's residuals rather than
the data, so there is no averaging to save it — past the point where signal runs out, the
extra trees fit noise. The rise is small, so it is worth checking fold by fold rather than
trusting the means.

In [10]:
# The U is only 0.41 cycles deep against a fold std of ~0.8, so compare the two settings
# within each fold, where the fold-to-fold variation cancels out.
paired = {}
for n_estimators in [200, 1000]:
    folds = evaluate.cross_validate_model(
        models.gradient_boosting(n_estimators=n_estimators), train, features
    )
    paired[n_estimators] = folds[folds["split"] == "validation"].set_index("fold")["rmse"]

paired = pd.DataFrame(paired)
paired["difference"] = paired[1000] - paired[200]
paired.round(3)

,200,1000,difference
fold,,,
1,25.665,26.162,0.496
2,23.833,24.181,0.348
3,25.056,25.521,0.464
4,23.819,24.122,0.303
5,24.661,25.077,0.416


Worse at 1000 estimators in every fold. The magnitude sits inside the noise but the
direction is consistent, so 200 is a real choice rather than a coin flip.

## 6. Final test evaluation

Fit on all 100 training engines, predict the last observed cycle of each of the 100 unseen
test engines, score against `RUL_FD001.txt` clipped the same way.

In [11]:
test_metrics, test_predictions = evaluate.evaluate_on_test(
    models.all_models(), train, test, features
)
test_metrics.round(3)

,model,rmse,mae,r2,phm08
0,Linear Regression,23.898,19.127,0.669,2590.421
1,Random Forest,21.894,16.086,0.722,2700.383
2,Gradient Boosting,22.218,15.898,0.714,2856.655


Every model scores 2–3 cycles better here than in cross-validation, and that is **not**
better generalization. The test set scores one mid-life snapshot per engine; the validation
folds score every cycle, including the near-failure ones where prediction is hardest. The
giveaway is R², which does not improve — it drops slightly. If these engines were genuinely
easier to generalize to, R² would rise as RMSE fell. The two numbers measure different
things and should not be compared directly.

### The PHM08 score

RMSE charges the same for a prediction 20 cycles early as 20 cycles late. Maintenance does
not: pulling an engine early wastes life, leaving it late risks failure. PHM08 encodes that
asymmetry — `exp(-d/13) - 1` for early predictions against `exp(d/10) - 1` for late ones,
where the smaller divisor makes the late branch climb faster.

In [12]:
print(f"20 cycles early: {evaluate.phm08_score([100], [80]):.2f}")
print(f"20 cycles late:  {evaluate.phm08_score([100], [120]):.2f}")

# It ranks linear regression best, against RMSE. Worth checking where that score comes from.
import numpy as np

for name, prediction in test_predictions.items():
    d = prediction - test["RUL"].to_numpy()
    penalty = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    worst_five = np.sort(penalty)[::-1][:5].sum()
    print(
        f"{name:18} total {penalty.sum():7.0f}  late {(d >= 0).mean():.0%} of engines  "
        f"worst 5 engines = {worst_five / penalty.sum():.0%} of score"
    )

20 cycles early: 3.66
20 cycles late:  6.39
Linear Regression  total    2590  late 72% of engines  worst 5 engines = 59% of score
Random Forest      total    2700  late 68% of engines  worst 5 engines = 73% of score
Gradient Boosting  total    2857  late 66% of engines  worst 5 engines = 74% of score


PHM08 ranks linear regression first, disagreeing with every other metric. The reason is
visible above: all three models lean late on this snapshot, and because the score is a sum
of exponentials it is dominated by a few engines — the worst five account for most of each
total. With 100 engines, that is a ranking decided by about five of them, and it should not
override RMSE here.

## Summary

Ranked on test RMSE: random forest (21.89), gradient boosting (22.22), linear regression
(23.90). The two tree models are tied within noise and both beat the baseline by about two
cycles.

The useful finding is not the ranking but the diagnosis behind it. Linear regression's
error is bias — it fits a plane to a curved degradation path, flattens where the signal
dies, and predicts negative remaining life. Clipping the target at 150 cut away the region
that was never learnable and revealed that the apparent overfitting was nothing of the
kind. Moving to trees addressed the actual problem, and the two ensembles reached the same
accuracy by opposite routes: the forest averaging deep overfitted trees into stability,
boosting stacking shallow corrections until it begins fitting noise.

Caveats are in the README's limitations section — simulated data, FD001's single operating
condition, and 100 engines, which is few enough that sub-cycle differences are noise.